# DS-01 Part 1 — Data understanding, EDA and feature engineering

Run from the repository root. The first cell downloads the public IBM Telco dataset if it is not already present. The notebook is for exploration; the reusable pipeline lives in `src/`.


In [ ]:
from src.data_access import load_raw_dataset
from src.quality import inspect_quality, clean_dataset
from src.feature_engineering import add_features
from src.config import REPORTS_DIR, FIGURES_DIR
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

raw = load_raw_dataset()
print('Shape:', raw.shape)
display(raw.head())

## 1. Data quality
Inspect schema, missing values, duplicate records and the target balance before cleaning.


In [ ]:
quality = inspect_quality(raw)
quality

In [ ]:
raw.info()
display(raw.isna().sum().sort_values(ascending=False).to_frame('missing_cells'))
display(raw['Churn'].value_counts(dropna=False).to_frame('customers'))

## 2. Clean data and engineer interpretable features
The raw dataset is not modified. Blank `TotalCharges` is set to 0 only when `tenure == 0`; unexpected blanks raise an error.


In [ ]:
clean = clean_dataset(raw)
features = add_features(clean)
print('Clean shape:', clean.shape)
print('Feature shape:', features.shape)
display(features[['customerID','tenure','MonthlyCharges','TotalCharges','Churn','service_count','tenure_group','is_month_to_month','is_electronic_check']].head(10))

## 3. Exploratory data analysis
These rates are descriptive associations, not causal effects.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
churn_counts = features['Churn'].map({0:'Stayed', 1:'Churned'}).value_counts().reindex(['Stayed', 'Churned'])
axes[0].bar(churn_counts.index, churn_counts.values)
axes[0].set(title='Churn class distribution', xlabel='Outcome', ylabel='Customers')
contract_rate = features.groupby('Contract', observed=False)['Churn'].mean().sort_values(ascending=False)
axes[1].bar(contract_rate.index.astype(str), contract_rate.values)
axes[1].set(title='Observed churn rate by contract', xlabel='Contract', ylabel='Churn rate')
axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
tenure_rate = features.groupby('tenure_group', observed=False)['Churn'].mean().dropna()
payment_rate = features.groupby('PaymentMethod', observed=False)['Churn'].mean().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(tenure_rate.index.astype(str), tenure_rate.values)
axes[0].set(title='Observed churn by tenure group', xlabel='Tenure group', ylabel='Churn rate')
axes[1].bar(payment_rate.index.astype(str), payment_rate.values)
axes[1].set(title='Observed churn by payment method', xlabel='Payment method', ylabel='Churn rate')
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=features, x=features['Churn'].map({0:'Stayed', 1:'Churned'}), y='MonthlyCharges', ax=axes[0])
axes[0].set(title='Monthly charges by churn label', xlabel='Outcome', ylabel='Monthly charges')
sns.scatterplot(data=features.sample(min(2000, len(features)), random_state=42), x='tenure', y='MonthlyCharges', hue=features.loc[features.sample(min(2000, len(features)), random_state=42).index, 'Churn'].map({0:'Stayed',1:'Churned'}), alpha=.6, ax=axes[1])
axes[1].set(title='Tenure vs monthly charges', xlabel='Tenure (months)', ylabel='Monthly charges')
plt.tight_layout()
plt.show()

## 4. Business and data limitations
- This is a customer snapshot, not a monthly sequence, so it cannot directly support monthly usage trend analysis or time-to-churn labels.
- There is no complaints/ticket field. `TechSupport` is a service subscription indicator, not a complaint measure.
- `TotalCharges` is cumulative billing, not profit/margin.
- `customerID` is an identifier and must not be used as a model feature.
- Differences in churn rates are associations, not proof of causal effects.

Run `python -m src.prepare_data` to save prepared CSVs, and `python -m src.run_eda` to generate the full report/charts.
